# NutriTrack — CSE476 CA2 Multi-Agent Demonstration

This runnable notebook demonstrates **NutriTrack**, a **Manager-Worker Multi-Agent Personalized Nutrition Tracking and Meal Planning System** built on the **Microsoft Agent Framework (`agent-framework`)**.

It covers:
1. Environment setup, Microsoft Agent Framework inspection, and LLM/offline mode detection.
2. Construction of all **5 separate MAF agents** and their **22 deterministic Python tools**.
3. Step-by-step **Manager-Worker Delegation** across all 5 core user scenarios.
4. Honest missing-food / missing-day / LLM-failure handling without fabricated data.
5. Full **Multi-Agent Execution Trace**, **Shared `NutritionState`**, and **Final Weekly Nutrition Report**.


In [1]:
from datetime import date, timedelta
import json
from pathlib import Path
import sys
import tempfile

# Resolve repository root reliably whether Jupyter starts in project root or notebooks/
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "nutritrack").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import agent_framework as af
from nutritrack.agents import (
    FailingChatClient,
    NutriTrackOrchestrator,
    create_chat_client,
    create_goal_budget_agent,
    create_meal_intake_agent,
    create_meal_recommendation_agent,
    create_nutrition_analysis_agent,
    create_nutrition_manager_agent,
    detect_llm_config,
)
from nutritrack.state import RunLog, StateManager
from nutritrack.tools import new_context
from nutritrack.tools.base import LLMUnavailableError

print(f"Project Root            : {PROJECT_ROOT}")
print(f"Microsoft AgentFramework: v{af.__version__}")
print(f"LLM Config Detected     : {detect_llm_config()}")


Project Root            : F:\downloads\NutriTrack
Microsoft AgentFramework: v1.26.0
LLM Config Detected     : {'mode': 'offline_demo', 'provider': 'local_deterministic', 'model': 'deterministic-tool-planner-v1', 'base_url': None, 'is_llm': False, 'has_key': False}


## 1. Initialize Shared `NutritionState`, `RunLog`, and All Five MAF Agents

In [2]:
# Isolated temporary directory so repeated runs of this notebook remain reproducible
demo_dir = tempfile.TemporaryDirectory()
state_manager = StateManager(base_dir=demo_dir.name, ca1_log_path=None)
run_log = RunLog(path=Path(demo_dir.name) / "agent_runs.jsonl")
orchestrator = NutriTrackOrchestrator(state_manager=state_manager, run_log=run_log)
mode_info = orchestrator.get_mode_info()
client = orchestrator.ai_manager.chat_client

inspect_ctx = new_context(state_manager, run_log, user_id="demo_user", task="agent_inspection")
agents = [
    create_nutrition_manager_agent(inspect_ctx, client=client),
    create_meal_intake_agent(inspect_ctx, client=client),
    create_nutrition_analysis_agent(inspect_ctx, client=client),
    create_goal_budget_agent(inspect_ctx, client=client),
    create_meal_recommendation_agent(inspect_ctx, client=client),
]

print(f"Execution mode: {mode_info['mode']} ({mode_info['provider']})\n")
for a in agents:
    tools = [t.name for t in (a.default_options.get("tools") or [])]
    print(f"- {a.name} (id={a.id})")
    print(f"  Role : {a.description}")
    print(f"  Tools ({len(tools)}): {tools}\n")

Execution mode: offline_demo (local_deterministic)

- Nutrition Manager Agent (id=nutrition_manager)
  Role : Coordinates user nutrition requests through Manager-Worker delegation, validates specialist outputs against shared state, handles failures, and compiles final responses.
  Tools (5): ['read_shared_state', 'delegate_to_specialist', 'validate_agent_results', 'update_workflow_status', 'append_run_log']

- Meal Intake Agent (id=meal_intake)
  Role : Parses natural-language food logs, validates quantities, looks up nutrition, and logs or removes meals.
  Tools (5): ['parse_meal_entry', 'lookup_food_nutrition', 'add_meal_to_state', 'remove_meal_from_state', 'register_custom_food']

- Nutrition Analysis Agent (id=nutrition_analysis)
  Role : Calculates daily nutrition totals, macro percentages, 7-day trends, unusual days, and target comparisons.
  Tools (4): ['calculate_daily_totals', 'calculate_macro_split', 'get_weekly_trends', 'compare_intake_with_targets']

- Goal and Budget Agent

## 2. Helper to Print Full Multi-Agent Execution Trace

Each turn captures:
- Manager workflow transitions (`planning` → `delegating` → `validating` → `completed` / `failed`)
- Selectively activated specialist agents
- Every Python tool invocation executed through MAF's `FunctionInvocationLayer`
- Shared `NutritionState` totals and remaining budget

In [3]:
def run_and_trace(prompt: str, *, user_id: str = "demo_user") -> dict:
    res = orchestrator.run_turn(prompt, user_id=user_id)
    print("=" * 84)
    print(f"USER REQUEST : {prompt}")
    print(f"RUN ID       : {res['run_id']}")
    print(f"STATUS       : {res['status']} | WORKFLOW: {res['workflow_status']}")
    print(f"DELEGATED TO : {res['activated_specialists'] or ['(None - Manager only)']}")
    print("-" * 84)
    print("EXECUTION TRACE:")
    for entry in res["run_log_entries"]:
        err_part = f" | ERROR={entry['error']['code']}: {entry['error']['message']}" if entry.get("error") else ""
        print(
            f"  [{entry['agent']}] {entry['tool']} -> {entry['status']}"
            f" | {entry.get('output_summary') or entry.get('input_summary') or ''}{err_part}"
        )
    print("-" * 84)
    print(f"FINAL RESPONSE:\n{res['response']}")
    snap = res["state"]
    print("-" * 84)
    print(
        f"SHARED STATE ({snap['date']}): "
        f"Consumed={snap['consumed']} | Targets={snap['targets']} | Remaining={snap['remaining']}"
    )
    print("=" * 84 + "\n")
    return res


## 3. Execute Core CA2 Scenarios (Manager-Worker Delegation)

In [4]:
# Scenario 1: Log food intake -> Manager activates Intake + Analysis + Goal/Budget
r1 = run_and_trace("I ate 3 boiled eggs and 2 bananas.")

# Scenario 2: Query consumed totals -> Manager activates Nutrition Analysis ONLY
r2 = run_and_trace("How many calories and protein did I eat today?")

# Scenario 3: Adjust goals -> Manager activates Goal & Budget ONLY
r3 = run_and_trace("Set my daily calorie goal to 2100 kcal and protein goal to 140 g.")

# Scenario 4: Recommend next meal -> Manager activates Goal & Budget + Meal Recommendation
r4 = run_and_trace("What meal should I eat next to hit my remaining calorie and protein target?")


USER REQUEST : I ate 3 boiled eggs and 2 bananas.
RUN ID       : run-20261115-000801-ef76414fe3
STATUS       : success | WORKFLOW: completed
DELEGATED TO : ['meal_intake', 'nutrition_analysis', 'goal_budget']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: idle -> planning.","previous":"idle","current":"planning","detail":"I ate 3 boiled eggs and 2 bananas."}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 0 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":0,"targets_source":{"calories":"demo_default","protein":"demo_default"}}
  [Meal Intake Agent] parse_meal_entry -> success | {"status":"success","code":"ok","message":"Found 2 food item(s).","items":"<2 entries>","meal_type":null,"date_hint":nu

USER REQUEST : How many calories and protein did I eat today?
RUN ID       : run-20261115-000801-47b75103b1
STATUS       : success | WORKFLOW: completed
DELEGATED TO : ['nutrition_analysis']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: completed -> planning.","previous":"completed","current":"planning","detail":"How many calories and protein did I eat today?"}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 2 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":2,"targets_source":{"calories":"demo_default","protein":"demo_default"}}
  [Nutrition Analysis Agent] calculate_daily_totals -> success | {"status":"success","code":"ok","message":"2026-11-15: 444 kcal, 20.6 g protein, 55.8 g carbs, 16.7 g 

USER REQUEST : Set my daily calorie goal to 2100 kcal and protein goal to 140 g.
RUN ID       : run-20261115-000801-7ec0949410
STATUS       : success | WORKFLOW: completed
DELEGATED TO : ['goal_budget']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: completed -> planning.","previous":"completed","current":"planning","detail":"Set my daily calorie goal to 2100 kcal and protein goal to 140 g."}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 2 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":2,"targets_source":{"calories":"demo_default","protein":"demo_default"}}
  [Goal and Budget Agent] set_calorie_target -> success | {"status":"success","code":"ok","message":"Daily calories target set to 2100 k

USER REQUEST : What meal should I eat next to hit my remaining calorie and protein target?
RUN ID       : run-20261115-000802-08cdf3b875
STATUS       : success | WORKFLOW: completed
DELEGATED TO : ['goal_budget', 'meal_recommendation']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: completed -> planning.","previous":"completed","current":"planning","detail":"What meal should I eat next to hit my remaining calorie and protein target?"}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 2 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":2,"targets_source":{"calories":"user","protein":"user"}}
  [Goal and Budget Agent] calculate_remaining_targets -> success | {"status":"success","code":"ok","message":

## 4. Weekly Nutrition Insights (Honest Multi-Day & Missing-Day Analytics)

In [5]:
# Seed realistic prior-day logs leaving intentional gaps to verify missing days are NOT counted as zero
st = state_manager.get("demo_user")
today = date.today()
st.add_meals(
    [{"food": "oats", "quantity_str": "100 g", "calories": 389, "protein": 16.9, "carbs": 66.3, "fat": 6.9},
     {"food": "chicken breast", "quantity_str": "200 g", "calories": 330, "protein": 62.0, "carbs": 0.0, "fat": 7.2},
     {"food": "rice", "quantity_str": "200 g", "calories": 260, "protein": 5.4, "carbs": 56.4, "fat": 0.6}],
    day=(today - timedelta(days=4)).isoformat(),
)
st.add_meals(
    [{"food": "banana", "quantity_str": "1 piece", "calories": 105, "protein": 1.3, "carbs": 27.0, "fat": 0.4}],
    day=(today - timedelta(days=2)).isoformat(),  # Low-intake day (<50% target)
)
st.add_meals(
    [{"food": "paneer", "quantity_str": "200 g", "calories": 530, "protein": 36.6, "carbs": 7.2, "fat": 40.0},
     {"food": "roti", "quantity_str": "4 pieces", "calories": 480, "protein": 14.0, "carbs": 92.0, "fat": 6.0},
     {"food": "whey protein", "quantity_str": "2 scoop", "calories": 240, "protein": 48.0, "carbs": 6.0, "fat": 3.0}],
    day=(today - timedelta(days=1)).isoformat(),
)

# Scenario 5: Weekly insights -> Manager activates Nutrition Analysis ONLY
r5 = run_and_trace("Show my weekly nutrition insights.")


USER REQUEST : Show my weekly nutrition insights.
RUN ID       : run-20261115-000802-490a7c71b8
STATUS       : success | WORKFLOW: completed
DELEGATED TO : ['nutrition_analysis']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: completed -> planning.","previous":"completed","current":"planning","detail":"Show my weekly nutrition insights."}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 2 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":2,"targets_source":{"calories":"user","protein":"user"}}
  [Nutrition Analysis Agent] calculate_daily_totals -> success | {"status":"success","code":"ok","message":"2026-11-15: 444 kcal, 20.6 g protein, 55.8 g carbs, 16.7 g fat from 2 item(s).","date":"2026-11-15"

## 5. Failure Handling & Honest Boundary Cases (Unknown Food + Simulated LLM Outage)

In [6]:
# Edge Case A: Unknown food -> never fabricates calories; prompts user instead
r_unknown = run_and_trace("I ate 1 dragonfruit")
print("Open Questions Recorded:", json.dumps(r_unknown["open_questions"], indent=2))

# Edge Case B: Simulated specialist LLM failure -> fails transparently without inventing recommendations
failing_client = FailingChatClient(
    error=LLMUnavailableError("Simulated provider timeout"),
    fail_for_agents={"meal_recommendation"},
    fallback_client=client,
)
fail_res = orchestrator.run_turn(
    "What meal should I eat next to hit my remaining calorie and protein target?",
    client=failing_client,
)
print("\n[LLM FAILURE TEST]")
print("Status         :", fail_res["status"])
print("Workflow Status:", fail_res["workflow_status"])
print("Error Code     :", fail_res["error_code"])
print("Manager Reply  :", fail_res["response"])


USER REQUEST : I ate 1 dragonfruit
RUN ID       : run-20261115-000802-b9a62539b8
STATUS       : needs_input | WORKFLOW: completed
DELEGATED TO : ['meal_intake', 'nutrition_analysis', 'goal_budget']
------------------------------------------------------------------------------------
EXECUTION TRACE:
  [Nutrition Manager Agent] update_workflow_status -> success | {"status":"success","code":"ok","message":"Workflow status: completed -> planning.","previous":"completed","current":"planning","detail":"I ate 1 dragonfruit"}
  [Nutrition Manager Agent] read_shared_state -> success | {"status":"success","code":"ok","message":"State for 2026-11-15: 2 meal entries, workflow 'planning'.","date":"2026-11-15","state":"<5 fields>","meal_count":2,"targets_source":{"calories":"user","protein":"user"}}
  [Meal Intake Agent] parse_meal_entry -> success | {"status":"success","code":"ok","message":"Found 1 food item(s).","items":"<1 entries>","meal_type":null,"date_hint":null,"ignored_phrases":[],"has_pro


[LLM FAILURE TEST]
Status         : failed
Workflow Status: failed
Error Code     : llm_unavailable
Manager Reply  : Unable to complete your request safely because a specialist failed (llm_unavailable): Meal Recommendation Agent could not complete the task (LLMUnavailableError: Simulated provider timeout).


## 6. Final Nutrition Report & Cleanup

In [7]:
final_snapshot = state_manager.get("demo_user").snapshot(history_days=7)
all_logs = run_log.read()

print("FINAL NUTRITION REPORT (demo_user)")
print(f"  Date                 : {final_snapshot['date']}")
print(f"  Targets              : {final_snapshot['targets']} (source={final_snapshot['targets_source']})")
print(f"  Consumed Today       : {final_snapshot['consumed']}")
print(f"  Remaining Budget     : {final_snapshot['remaining']}")
print(f"  Meals Today          : {len(final_snapshot['meals'])}")
for m in final_snapshot["meals"]:
    print(f"    - {m['quantity_str']} {m['food']}: {m['calories']} kcal, {m['protein']}g protein")
print(f"  Saved Recommendations: {len(final_snapshot['recommendations'])}")
if final_snapshot["recommendations"]:
    top = final_snapshot["recommendations"][0]
    print(f"    Top Pick: {top['name']} ({top['calories']} kcal, {top['protein']}g P)")
print(f"  Days With History    : {len(final_snapshot['weekly_history'])} recorded day(s)")
print(f"  Total Log Entries    : {len(all_logs)} entries written to append-only JSONL trace")

demo_dir.cleanup()


FINAL NUTRITION REPORT (demo_user)
  Date                 : 2026-11-15
  Targets              : {'calories': 2100.0, 'protein': 140.0} (source={'calories': 'user', 'protein': 'user'})
  Consumed Today       : {'calories': 444.0, 'protein': 20.6, 'carbs': 55.8, 'fat': 16.7}
  Remaining Budget     : {'calories': 1656.0, 'protein': 119.4}
  Meals Today          : 2
    - 3 pieces egg: 234.0 kcal, 18.0g protein
    - 2 pieces banana: 210.0 kcal, 2.6g protein
  Saved Recommendations: 1
    Top Pick: 200.0 g chicken breast + 200.0 g brown rice + 1 piece carrot (601.0 kcal, 68.0g P)
  Days With History    : 4 recorded day(s)
  Total Log Entries    : 88 entries written to append-only JSONL trace
